# 4. The Python a Biologist Needs

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `seqtools.py`

Small tools for DNA sequences, written from scratch.


In [ ]:
"""Small tools for DNA sequences, written from scratch."""

PAIRS = {"A": "T", "T": "A", "C": "G", "G": "C", "N": "N"}

# The standard genetic code, as NCBI publishes it: the amino acids for
# all 64 codons, with the letters of each codon taken in the order TCAG.
BASES = "TCAG"
AMINO_ACIDS = (
    "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
)

GENETIC_CODE = {}
for first in BASES:
    for second in BASES:
        for third in BASES:
            GENETIC_CODE[first + second + third] = AMINO_ACIDS[
                len(GENETIC_CODE)
            ]


def reverse_complement(dna: str) -> str:
    """The other strand, read 5' to 3'."""
    return "".join(PAIRS[base] for base in reversed(dna))


def gc_content(dna: str) -> float:
    """The share of the sequence that is G or C, from 0 to 1."""
    return (dna.count("G") + dna.count("C")) / len(dna)


def translate(dna: str) -> str:
    """Translate from the first letter until the first stop codon."""
    protein = []
    for start in range(0, len(dna) - 2, 3):
        amino_acid = GENETIC_CODE[dna[start : start + 3]]
        if amino_acid == "*":
            break
        protein.append(amino_acid)
    return "".join(protein)

## `fasta.py`

A FASTA reader, written from scratch.


In [ ]:
"""A FASTA reader, written from scratch."""


def read_fasta(path: str) -> dict[str, str]:
    """Map each record's header line to its sequence, in capitals."""
    sequences = {}
    header = None
    with open(path) as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                header = line[1:]
                sequences[header] = []
            elif header is None:
                raise ValueError(f"{path}: sequence before any header")
            else:
                sequences[header].append(line.upper())
    return {header: "".join(lines) for header, lines in sequences.items()}


def gene_name(header: str) -> str:
    """The value of [gene=...] in an NCBI header, or its first word."""
    if "[gene=" not in header:
        return header.split()[0]
    return header.split("[gene=")[1].split("]")[0]

## `globins.py`

Three globin coding sequences from NCBI: human HBB (NM_000518.5), human HBA1 (NM_000558.5) and mouse Hbb-bt (NM_008220.5).


In [ ]:
# Three globin coding sequences from NCBI: human HBB (NM_000518.5),
# human HBA1 (NM_000558.5) and mouse Hbb-bt (NM_008220.5).
from fasta import gene_name, read_fasta
from seqtools import gc_content, reverse_complement, translate

genes = read_fasta("globin_cds.fasta")

for header, dna in genes.items():
    protein = translate(dna)
    print(gene_name(header))
    print(f"  {len(dna)} bases, GC {gc_content(dna):.1%}")
    print(f"  protein: {len(protein)} amino acids, {protein[:12]}...")
    print(f"  other strand starts {reverse_complement(dna)[:12]}...")

## `check_uniprot.py`

Do our translations match the proteins UniProt lists (Chapter 10)?


In [ ]:
# Do our translations match the proteins UniProt lists (Chapter 10)?
from fasta import gene_name, read_fasta
from seqtools import translate

genes = read_fasta("globin_cds.fasta")
proteins = read_fasta("../ch10/globins.fasta")

expected = {"HBB": "P68871", "HBA1": "P69905", "Hbb-bt": "P02088"}
uniprot = {header.split("|")[1]: seq for header, seq in proteins.items()}

for header, dna in genes.items():
    name = gene_name(header)
    ours, theirs = translate(dna), uniprot[expected[name]]
    print(f"{name:7} against {expected[name]}: ", end="")
    if ours == theirs:
        print("identical")
        continue
    print("differs at")
    for position, (a, b) in enumerate(zip(ours, theirs), start=1):
        if a != b:
            print(f"  position {position:3}: ours {a}, UniProt {b}")